## 1. Introduction: Modular Ensemble Inference

This notebook performs a three-stage inference process to reconstruct the urban habitat map:
1. **Individual Inference:** Load three separate SegFormer models (Built-up, Grassland, High-Veg).
2. **Binary Generation:** Run each model over the test set and save binary probability masks.
3. **Hierarchical Synthesis:** Use the priority logic (High-Veg > Grassland > Built-up) to merge predictions.
4. **Ensemble Evaluation:** Calculate mIoU and Accuracy by comparing the synthesized result against the multi-class ground truth.

In [ ]:
import os
import torch
import numpy as np
import rasterio
from tqdm.auto import tqdm
from PIL import Image
from torch import nn
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassJaccardIndex, MulticlassAccuracy

# --- CONFIGURATION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DATA = "../01_data/processed_modular/val" # Using 'val' as our test set
MODEL_DIR = "../02_models/checkpoints"

# Paths to your saved .pt or HuggingFace model folders
CONFIG = {
    "builtup":  {"path": f"{MODEL_DIR}/segformer_builtup_final", "class_id": 1},
    "grassland": {"path": f"{MODEL_DIR}/segformer_grassland_final", "class_id": 2},
    "highveg":   {"path": f"{MODEL_DIR}/segformer_highveg_final", "class_id": 3}
}

IMG_SIZE = 512
OUTPUT_DIR = "../03_predictions/hierarchical_ensemble"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Model Initialization
We load the models into a dictionary for easy iteration. We assume each model was trained as a binary classifier (2 output channels: Background vs. Class).

In [ ]:
processor = SegformerImageProcessor.from_pretrained("nvidia/mit-b0")
models = {}

for task, info in CONFIG.items():
    print(f"Loading {task} model from {info['path']}...")
    models[task] = SegformerForSemanticSegmentation.from_pretrained(info['path'])
    models[task].to(DEVICE)
    models[task].eval()

## 3. Step-wise Inference & Synthesis
For each image in the test set, we run all three models, threshold their logits at 0.5, and then apply the priority rules: **High-Veg > Grassland > Built-up**.

In [ ]:
image_filenames = [f for f in os.listdir(f"{BASE_DATA}/images") if f.endswith(".tif")]
results_list = []

def get_binary_prediction(model, image_path):
    # Load and process image
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        # Rescale logits to original image size (512x512)
        upsampled_logits = nn.functional.interpolate(
            logits, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False
        )
        # Get class with highest probability (binary: 0 or 1)
        pred = upsampled_logits.argmax(dim=1).cpu().numpy()[0]
    return pred

print(f"Running hierarchical inference on {len(image_filenames)} images...")

for fname in tqdm(image_filenames):
    uid = fname.replace("_swissimage.tif", "")
    img_path = f"{BASE_OUT}/val/images/{fname}"
    
    # 1. Individual Binary Inferences
    pred_built = get_binary_prediction(models["builtup"], img_path)
    pred_grass = get_binary_prediction(models["grassland"], img_path)
    pred_highveg = get_binary_prediction(models["highveg"], img_path)
    
    # 2. Hierarchical Synthesis (Priority Logic)
    # Start with Background (0)
    final_pred = np.zeros((IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    
    # Layer 1: Built-up
    final_pred[pred_built == 1] = 1
    # Layer 2: Grassland (Overwrites Built-up)
    final_pred[pred_grass == 1] = 2
    # Layer 3: High-Veg (Overwrites Everything)
    final_pred[pred_highveg == 1] = 3
    
    # 3. Save synthesized TIF
    out_path = f"{OUTPUT_DIR}/{uid}_ensemble_pred.tif"
    with rasterio.open(img_path) as src:
        meta = src.meta.copy()
        meta.update(dtype=rasterio.uint8, count=1)
        with rasterio.open(out_path, 'w', **meta) as dst:
            dst.write(final_pred, 1)

## 4. Scoring the Ensemble
We compare our synthesized multi-class predictions against the ground truth labels we generated in Notebook 5.

In [ ]:
# Initialize metrics for 4 classes (0, 1, 2, 3)
miou_metric = MulticlassJaccardIndex(num_classes=4, average='macro').to(DEVICE)
acc_metric = MulticlassAccuracy(num_classes=4, average='macro').to(DEVICE)

all_preds = []
all_labels = []

print("Loading results for scoring...")
for fname in image_filenames:
    uid = fname.replace("_swissimage.tif", "")
    pred_path = f"{OUTPUT_DIR}/{uid}_ensemble_pred.tif"
    label_path = f"{BASE_DATA}/masks_combined/{uid}_mask.tif"
    
    with rasterio.open(pred_path) as s: all_preds.append(s.read(1))
    with rasterio.open(label_path) as s: all_labels.append(s.read(1))

# Convert to tensors for scoring
preds_tensor = torch.from_numpy(np.array(all_preds)).to(DEVICE)
labels_tensor = torch.from_numpy(np.array(all_labels)).to(DEVICE)

miou = miou_metric(preds_tensor, labels_tensor)
acc = acc_metric(preds_tensor, labels_tensor)

print(f"\n--- ENSEMBLE PERFORMANCE ---")
print(f"mIoU (Hierarchical): {miou.item():.4f}")
print(f"Overall Accuracy:     {acc.item():.4f}")

## 5. Prediction Visualization
Final check of the ensemble output.

In [ ]:
from matplotlib.colors import ListedColormap
custom_cmap = ListedColormap(['#d3d3d3', '#e74c3c', '#2ecc71', '#1b5e20']) # Gray, Red, Grass, Tree

# Pick a sample
idx = np.random.randint(0, len(image_filenames))
fname = image_filenames[idx]
uid = fname.replace("_swissimage.tif", "")

img = Image.open(f"{BASE_DATA}/images/{fname}")
pred = rasterio.open(f"{OUTPUT_DIR}/{uid}_ensemble_pred.tif").read(1)
label = rasterio.open(f"{BASE_DATA}/masks_combined/{uid}_mask.tif").read(1)

fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(img)
ax[0].set_title("Original Image")
ax[1].imshow(label, cmap=custom_cmap, vmin=0, vmax=3)
ax[1].set_title("Ground Truth (Combined)")
ax[2].imshow(pred, cmap=custom_cmap, vmin=0, vmax=3)
ax[2].set_title("Ensemble Prediction")
plt.show()